# IIP314W Optimización Aplicada a Negocios - 2026-T1
## Ayudantía 5: Big M y Variables de Activación

### Ejercicio 1: Localización de Plantas con Restricciones de Activación

Una empresa forestal chilena debe decidir qué plantas abrir de un conjunto de ubicaciones potenciales para abastecer a ciudades distribuidas por todo el país. Cada planta tiene un costo fijo anual por su operación (principalmente instalaciones, administración y mantenimiento). Si la planta está abierta, puede producir y transportar hasta su capacidad máxima a las diferentes ciudades; si está cerrada, no puede transportar nada. El costo de transporte depende de la planta de origen y la ciudad destino. Cada ciudad tiene una demanda fija que debe satisfacerse en su totalidad desde una o más plantas.

De manera adicional, la empresa enfrenta una restricción presupuestaria: por limitaciones energéticas de la región, puede abrir a lo más 2 plantas simultáneamente. Además, si una planta está abierta, debe ser utilizada por lo menos en el 30% de su capacidad (para justificar su operación).

**Parámetros disponibles:**
- Plantas potenciales: $I$
- Ciudades clientes: $J$
- Capacidad máxima de planta $i$: $u_i$
- Costo fijo anual de abrir planta $i$: $f_i$
- Costo variable de transporte por unidad desde planta $i$ a ciudad $j$: $c_{ij}$
- Demanda de ciudad $j$: $d_j$

**Se pide:**

**a) Modelamiento:**

Formule un modelo de Programación Lineal Entera (MIP) que minimice los costos totales. Defina claramente:
- Conjuntos
- Parámetros
- Variables de decisión (identifique cuáles son binarias y por qué)
- Función objetivo
- Restricciones (identifique qué restricciones usan la técnica de Big M y justifique)

#### Espacio para modelamiento:
(Indique conjuntos, parámetros, variables, FO y restricciones. Justifique el uso de Big M donde sea necesario)

---
### Ejercicio 2: Implementación Simple en Gurobi

Considere una versión simplificada del Ejercicio 1 (Localización de Plantas): 3 plantas potenciales ($I=3$) para abastecer 4 ciudades ($J=4$). La empresa puede abrir a lo más 2 plantas.

| Planta | Capacidad (unid) | Costo Fijo (\$) |
|:---:|:---:|:---:|
| A | 150 | 500 |
| B | 200 | 700 |
| C | 180 | 600 |

| Ciudad | Demanda (unid) |
|:---:|:---:|
| 1 | 80 |
| 2 | 100 |
| 3 | 90 |
| 4 | 70 |

Matriz de Costos de Transporte (\$/unid):

|  | Ciudad 1 | Ciudad 2 | Ciudad 3 | Ciudad 4 |
|:---:|:---:|:---:|:---:|:---:|
| Planta A | 12 | 15 | 18 | 20 |
| Planta B | 14 | 10 | 12 | 16 |
| Planta C | 16 | 14 | 8 | 11 |

**Se pide:**

**a) Modelamiento:**
Formule el MIP que minimice costo total. Use:
- Variable binaria $y_i \in \{0,1\}$ para indicar si planta $i$ está abierta
- Variable continua $x_{ij} \geq 0$ para flujo desde planta $i$ a ciudad $j$
- Restricción: $\sum_i y_i \leq 2$ (máximo 2 plantas abiertas)
- Use la capacidad condicional: $\sum_j x_{ij} \leq \text{u}_i \cdot y_i$

**b) Implementación en Gurobi:**
Implemente y resuelva el modelo. Reporte:
- Valor óptimo de la función objetivo
- Cuáles plantas se abren
- El plan de transporte óptimo
- Desglose de costos (fijos vs variables)

In [1]:
import gurobipy as gp
from gurobipy import GRB

# Nombre de plantas y ciudades
plantas = ['Planta_A', 'Planta_B', 'Planta_C']
ciudades = [1, 2, 3, 4]

# Capacidades de plantas (unidades)
capacidad = {
    'Planta_A': 150,
    'Planta_B': 200,
    'Planta_C': 180
}

# Costos fijos de apertura (pesos)
costo_fijo = {
    'Planta_A': 500,
    'Planta_B': 700,
    'Planta_C': 600
}

# Demanda de ciudades (unidades)
demanda = {1: 80, 2: 100, 3: 90, 4: 70}

# Matriz de costos de transporte (pesos/unidad)
# Filas: plantas, Columnas: ciudades
costo_transporte = {
    ('Planta_A', 1): 12, ('Planta_A', 2): 15, ('Planta_A', 3): 18, ('Planta_A', 4): 20,
    ('Planta_B', 1): 14, ('Planta_B', 2): 10, ('Planta_B', 3): 12, ('Planta_B', 4): 16,
    ('Planta_C', 1): 16, ('Planta_C', 2): 14, ('Planta_C', 3): 8,  ('Planta_C', 4): 11,
}

# Parámetros
max_plantas_abiertas = 2
demanda_total = sum(demanda.values())
capacidad_total = sum(capacidad.values())
# b) Desarrollo con Gurobi



m = gp.Model("Facility_Location_Problem")

# Variables de decisión binarias: ¿está abierta la planta i?
y = {}
for p in plantas:
    y[p] = m.addVar(vtype=GRB.BINARY, name=f"abierta_{p}")
x = {}
for p in plantas:
    for c in ciudades:
        x[p, c] = m.addVar(lb=0, name=f"transport_{p}_to_{c}")


# Costo fijo total (apertura de plantas)
costo_fijo_total = gp.quicksum(costo_fijo[p] * y[p] for p in plantas)

# Costo variable total (transporte)
costo_variable_total = gp.quicksum(costo_transporte[p, c] * x[p, c] 
                                   for p in plantas for c in ciudades)

# Función objetivo: minimizar costo total
m.setObjective(costo_fijo_total + costo_variable_total, GRB.MINIMIZE)

# Restricción 1: Satisfacción de Demanda
# Cada ciudad debe recibir exactamente su demanda
for c in ciudades:
    m.addConstr(
        gp.quicksum(x[p, c] for p in plantas) == demanda[c],
        name=f"demanda_ciudad_{c}"
    )

print(f"✓ Restricción 1 (Demanda): {len(ciudades)} restricciones agregadas")

# Restricción 2: Capacidad Condicional a Activación
# El flujo saliente de planta i no puede exceder su capacidad si está abierta
# Si y_i = 0 (cerrada): suma_j(x_ij) <= 0, força x_ij = 0
# Si y_i = 1 (abierta): suma_j(x_ij) <= capacidad_i
for p in plantas:
    m.addConstr(
        gp.quicksum(x[p, c] for c in ciudades) <= capacidad[p] * y[p],
        name=f"capacidad_{p}"
    )

print(f"✓ Restricción 2 (Capacidad Condicional): {len(plantas)} restricciones agregadas")

# Restricción 3: Máximo de Plantas Abiertas Simultáneamente
# Por limitaciones energéticas, se pueden abrir a lo más K plantas
m.addConstr(
    gp.quicksum(y[p] for p in plantas) <= max_plantas_abiertas,
    name="max_plantas_abiertas"
)

print(f"✓ Restricción 3 (Máx Plantas): 1 restricción agregada")
print()

print(f"Total de restricciones generadas: {m.numConstrs}")
print()

m.optimize()
print()

# Verificar estado
if m.status == GRB.OPTIMAL:
    print("✓ SOLUCIÓN ÓPTIMA ENCONTRADA")
else:
    print(f"Estado de optimización: {m.status}")
    
print()

Set parameter Username
Academic license - for non-commercial use only - expires 2026-06-16
✓ Restricción 1 (Demanda): 4 restricciones agregadas
✓ Restricción 2 (Capacidad Condicional): 3 restricciones agregadas
✓ Restricción 3 (Máx Plantas): 1 restricción agregada

Total de restricciones generadas: 0

Gurobi Optimizer version 12.0.2 build v12.0.2rc0 (win64 - Windows 10.0 (19045.2))

CPU model: 11th Gen Intel(R) Core(TM) i7-11800H @ 2.30GHz, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 8 physical cores, 16 logical processors, using up to 16 threads

Optimize a model with 8 rows, 15 columns and 30 nonzeros
Model fingerprint: 0xa285dc43
Variable types: 12 continuous, 3 integer (3 binary)
Coefficient statistics:
  Matrix range     [1e+00, 2e+02]
  Objective range  [8e+00, 7e+02]
  Bounds range     [1e+00, 1e+00]
  RHS range        [2e+00, 1e+02]
Presolve time: 0.02s
Presolved: 8 rows, 15 columns, 30 nonzeros
Variable types: 12 continuous, 3 integer (3 binary)
Found heuristic soluti